# Exact (Hill) cat on the grid of the cat-start map, up to the cat fold (22/10/2026)

Companion of `Hill_satellite_seed_grid.ipynb`, for `lyapunov_heatmap_cat_start.ipynb`. The cat-start map starts from $a(0) = i|\alpha|$, $b(0) = 0$, which is not on the cat orbit,
and its chaos onset lies 6–9% below the cat fold (`Hill_method_cat_fold.ipynb`), where the basin of the cat shrinks. The question is whether a trajectory started **exactly on the cat**
becomes chaotic only at the fold, where the cat disappears with a saddle orbit.

**What this notebook does.** For every $\varepsilon_d/g_2$ of the map ($2{:}0.1{:}13.5$) the cat orbit is followed by pseudo-arclength from the bottom of the map ($\varepsilon_p = 0.2$) towards higher $g_2$,
up to its **fold**. The state at $\tau = 0$ is stored at every $\varepsilon_p$ of the map ($0.2{:}0.01{:}1.1$) below the fold; above the fold the cat does not exist and the entry is `null`
(the map then falls back on $a(0) = i|\alpha|$, $b(0) = 0$). Steps of the arclength that pass the fold (the parameter $s$ turns back) are rejected and halved, so the branch is never continued onto the saddle.
For $\varepsilon_d/g_2 \lesssim 6.4$ the fold is above the top of the map ($g_2^\ast|\alpha| \approx 219$ MHz) and the cat exists on the whole row.

**Start.** The hard-coded cat of `Hill_method_cat_fold` ($\varepsilon_p = 0.5$, $\varepsilon_d = 10.12\,g_2$, converged with the former $J_1$ definition of $g_2$ and continued in $\varepsilon_d$ to the $\sin$ definition),
continued in $\varepsilon_d/g_2$ at $\varepsilon_p = 0.5$, then down to $\varepsilon_p = 0.2$.

**Both cats.** The orbit shifted by half a period, $\mathbf X(\tau + \pi)$, is the other cat (the drive has period $\pi$; the memory harmonics change sign). `state_plus` is the cat near $a(0) = +i|\alpha|$ (the one of `cat_sign = 1`), `state_minus` the other one.

**Conventions** as in the other Hill notebooks: $g_2 = E_J\sin\varepsilon_p\,\varphi_a^2\varphi_b$, $\kappa_b = \pi/10.4$ (code), $\tau = \omega_a t$, $x^\lambda = \varphi x$, $N_H = 16$.
In Julia (`SystemATS`, $t = 0$): $a = (x_a + i\,dx_a/d\tau)/(\sqrt2\varphi_a)$, $b = (x_b + i\,(dx_b/d\tau)/2)/(\sqrt2\varphi_b)$.

**Output** `data/hill_orbits/cat_seed_hill_22_10_2026.json`. The whole notebook runs headless in about 5 min.

In [ ]:
import json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/pylib")
from catchaos.params import *                      # w_a, phi_a, phi_b, E_J, kappa_b (pi/10.4, code convention), G0 (g2 = s G0), MHz
from catchaos.plotting import *                    # Plot_setting (figsize, Perso_palette_float, ...)
from catchaos.ats import HB, full_rhs
from catchaos.hill import strobe, monodromy, branch_on_grid
from catchaos.seeds import cat_seed
from catchaos.jsonio import nan2none

OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/"

## Harmonic balance (`catchaos`, as in `Hill_satellite_seed_grid.ipynb` / `Hill_method_satellite_fold.ipynb`)
`catchaos.ats.HB`: `set_s(s, r)` sets $g_2 = sG_0$ and $\varepsilon_d = r\,g_2$. `strobe`, `monodromy` and `full_rhs` also come from `catchaos`.

## Cat seed and continuation up to the fold
The seed is `catchaos.seeds.cat_seed` (the hard-coded cat at $\varepsilon_p = 0.5$, continued from the $J_1$ to the $\sin$ definition of $g_2$).
The continuation is `catchaos.hill.branch_on_grid(..., direction=1)`: `satellite_column` of `Hill_satellite_seed_grid` run the other way, towards higher $s$, stopping at the first turning point.

## All columns of the map

In [ ]:
eps_p_list = np.round(np.arange(0.2, 1.1 + 1e-9, 0.01), 2)      # = Julia vcat(0.2:.01:1.1)
ratio_list = np.round(np.arange(2.0, 13.5 + 1e-9, 0.1), 1)      # = Julia 2:.1:13.5
s_grid = np.sin(eps_p_list)
i_half = np.argmin(np.abs(eps_p_list - 0.5))

N_H = 16
hb = HB(N_H)
z10 = cat_seed(hb)                                               # eps_p = 0.5, eps_d = 10.12 g2
z_bottom = {}                                                    # cat at eps_p = 0.2 for every ratio
for part in (ratio_list[ratio_list > 10.12], ratio_list[ratio_list < 10.12][::-1]):
    z, r_prev = z10.copy(), 10.12
    for r in part:
        for rr in np.linspace(r_prev, r, 3)[1:]:
            hb.set_s(s_grid[i_half], rr); z = hb.solve(z)
        r_prev = r
        zb = z.copy()
        for i in range(i_half - 1, -1, -1):
            hb.set_s(s_grid[i], r); zb = hb.solve(zb)
        z_bottom[r] = zb

In [ ]:
n_r, n_p = len(ratio_list), len(eps_p_list)
state_plus = np.full((n_r, n_p, 4), np.nan)       # [x_a, x_b, dx_a/dtau, dx_b/dtau] at tau = 0, cat near a(0) = +i|alpha|
state_minus = np.full((n_r, n_p, 4), np.nan)      # the other cat (orbit at tau = pi)
mu_all = np.full((n_r, n_p, 4), np.nan + 0j)
s_fold = np.full(n_r, np.nan)
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # overflows inside rejected Powell iterations
    for j, r in enumerate(ratio_list):
        zs, s_fold[j], _ = branch_on_grid(hb, lambda s: hb.set_s(s, r), z_bottom[r], s_grid[0], s_grid, direction=1)
        for i in np.where(np.isfinite(zs[:, 0]))[0]:
            y0, ypi = strobe(zs[i], N_H, 0.0), strobe(zs[i], N_H, np.pi)
            if ypi[2] > y0[2]:                       # Im a(0) ∝ dx_a/dtau: keep the cat with Im a(0) > 0 as "plus"
                y0, ypi = ypi, y0
            state_plus[j, i], state_minus[j, i] = y0, ypi
            hb.set_s(s_grid[i], r); mu_all[j, i] = monodromy(hb, zs[i])
g2_fold = MHz(s_fold*G0)
max_abs_mu = np.abs(mu_all).max(axis=2)
has = np.isfinite(state_plus[..., 0])
print(f"{time.time() - t0:.0f} s; cat on {has.sum()}/{has.size} grid points ({100*has.mean():.1f}%), stable (max|mu| < 1) on {(max_abs_mu < 1).sum()}")
print(f"cat fold inside the map for eps_d/g2 >= {ratio_list[np.isfinite(g2_fold)].min()}: "
      f"{np.nanmax(g2_fold):.2f} MHz .. {np.nanmin(g2_fold):.2f} MHz (eps_d/g2 = {ratio_list[-1]})")

## Checks
1. The fold against `cat_fold_line_19_10_2026.json` (`Hill_method_cat_fold`, $N_H = 12$, ratios $2{:}0.25{:}13.5$; compared where the two grids coincide).
2. The cat is stable on the whole stored range (max|μ| < 1), up to the last grid point below the fold.
3. The stored state is on the orbit: time integration of the full equations over one period, at the highest grid point of each column.
4. `state_plus` is the cat of `cat_sign = 1`: $a(0)$ close to $+i|\alpha|$, $|\alpha|^2 = \varepsilon_d/2g_2$.

In [ ]:
with open(OUT_DIR + "hill_orbits/cat_fold_line_19_10_2026.json") as f:
    ref = json.load(f)
ref_g = {round(r, 2): (np.nan if g is None else g) for r, g in zip(ref["eps_d_over_g2"], ref["g2_fold_MHz"])}
common = [j for j, r in enumerate(ratio_list) if round(r, 2) in ref_g]
d = np.array([g2_fold[j] - ref_g[round(ratio_list[j], 2)] for j in common])
top = MHz(s_grid[-1]*G0)                    # the reference is NaN only without a fold below g2_max = 137 MHz: compare with the top of the map
both_nan = np.array([np.isnan(g2_fold[j]) == (not ref_g[round(ratio_list[j], 2)] <= top) for j in common])
print(f"1. fold vs cat_fold_line_19_10_2026 at {len(common)} common ratios: max |difference| = {np.nanmax(np.abs(d)):.3f} MHz; "
      f"same set of ratios without a fold inside the map: {both_nan.all()}")

print(f"2. max|mu| on the stored cats: {np.nanmin(max_abs_mu):.3f} .. {np.nanmax(max_abs_mu):.4f}")

i_hi = n_p - 1 - np.argmax(has[:, ::-1], axis=1)                 # highest eps_p with a cat, per ratio
err = []
for j, r in enumerate(ratio_list):
    y0 = state_plus[j, i_hi[j]]
    y1 = solve_ivp(full_rhs(s_grid[i_hi[j]], r), (0, 2*np.pi), y0, method="DOP853", rtol=1e-11, atol=1e-13).y[:, -1]
    err.append(np.linalg.norm(y1 - y0)/np.linalg.norm(y0))
print(f"3. one-period return error, relative: max {np.max(err):.1e}")

a0 = (state_plus[..., 0] + 1j*state_plus[..., 2])/(np.sqrt(2)*phi_a)
alpha = np.sqrt(ratio_list/2)[:, None]
print(f"4. a(0)/(i|alpha|): real part {np.nanmin((a0/(1j*alpha)).real):.3f} .. {np.nanmax((a0/(1j*alpha)).real):.3f}, "
      f"arg/pi {np.nanmin(np.angle(a0/(1j*alpha)))/np.pi:+.3f} .. {np.nanmax(np.angle(a0/(1j*alpha)))/np.pi:+.3f}")

In [ ]:
g2_MHz_list = MHz(s_grid*G0)
alpha_sq = ratio_list/2
fig, axes = plt.subplots(1, 2, figsize=(2*figsize[0], figsize[1]))
ax = axes[0]
pc = ax.pcolormesh(g2_MHz_list, alpha_sq, np.abs(a0)/np.sqrt(alpha_sq)[:, None], cmap="viridis", shading="nearest")
fig.colorbar(pc, ax=ax, label=r"$|a(0)|/|\alpha|$")
ax.set_title("Seed: exact cat at $t = 0$", fontsize=8)
ax = axes[1]
pc = ax.pcolormesh(g2_MHz_list, alpha_sq, max_abs_mu, cmap="viridis", shading="nearest")
fig.colorbar(pc, ax=ax, label=r"$\max|\mu|$")
ax.set_title("Floquet stability of the cat", fontsize=8)
for ax in axes:
    ax.plot(g2_fold, alpha_sq, "k--", lw=1, label="cat fold (Hill)")
    ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\varepsilon_d/2g_2$")
    ax.set_xlim(g2_MHz_list.min(), g2_MHz_list.max()); ax.set_ylim(alpha_sq.min(), alpha_sq.max())
axes[0].legend(fontsize=5, loc="lower left")
plt.tight_layout()
plt.savefig(FIG_DIR + "cat_seed_hill_grid_22_10_2026.pdf")
plt.show()

## Export
`null` above the fold. Outer index: $\varepsilon_d/g_2$ (`j`), inner index: $\varepsilon_p$ (`i`), i.e. `state_plus[j][i]`, as `Lyap_coef_chaos[j, i]` in the Julia map. `g2_fold_MHz[j]` is `null` when the fold is above the map.

In [ ]:
with open(OUT_DIR + "hill_orbits/cat_seed_hill_22_10_2026.json", "w") as f:
    json.dump({"description": "Exact period-1 cat (harmonic balance, pseudo-arclength from eps_p = 0.2 up to its fold) "
                              "on the grid of lyapunov_heatmap_cat_start.ipynb; Hill_cat_seed_grid.ipynb",
               "g2_definition": "E_J sin(eps_p) phi_a^2 phi_b", "kappa_b": kappa_b, "N_H": N_H,
               "state_definition": "[x_a, x_b, dx_a/dtau, dx_b/dtau] at tau = omega_a t = 0, x^lambda = phi x; "
                                   "a = (x_a + i dx_a/dtau)/(sqrt2 phi_a), b = (x_b + i (dx_b/dtau)/2)/(sqrt2 phi_b); null above the fold. "
                                   "state_plus: cat with Im a(0) > 0 (cat_sign = 1), state_minus: the other cat",
               "eps_p_list": eps_p_list.tolist(), "eps_d_over_g2": ratio_list.tolist(), "g2_MHz": g2_MHz_list.tolist(),
               "state_plus": nan2none(state_plus), "state_minus": nan2none(state_minus), "max_abs_mu": nan2none(max_abs_mu),
               "s_fold": [None if np.isnan(v) else float(v) for v in s_fold],
               "g2_fold_MHz": [None if np.isnan(v) else float(v) for v in g2_fold]}, f)
print("written", OUT_DIR + "hill_orbits/cat_seed_hill_22_10_2026.json")